# Retail Sales Performance Analysis

## Project Overview

This project analyzes supermarket transaction data to evaluate sales performance, profitability, customer behavior, payment preferences, and sales trends.

The analysis uses the original dataset containing 1,000 transactions and applies PySpark for data processing, Pandas for analysis, and Plotly for visualization.

## Business Objective

The objective of this project is to identify important sales and customer patterns and translate the findings into actionable business recommendations.

## Business Questions

This analysis aims to answer the following questions:

1. Which city generates the highest sales?
2. Which product line performs best in terms of sales and gross income?
3. Which customer type contributes the most sales?
4. Which payment method is most frequently used?
5. How does sales performance change over time?
6. Is customer rating associated with transaction value?
7. What business areas should management prioritize

In [2]:
!pip install -q pyspark plotly

In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

def display_formatted(df_pandas, decimals=2, integer_columns=None, percent_columns=None):
    """Display analysis tables with consistent numeric formatting without changing raw values."""
    table = df_pandas.copy()
    integer_columns = integer_columns or []
    percent_columns = percent_columns or []

    for column in table.columns:
        if column in integer_columns:
            table[column] = table[column].map(lambda x: f"{int(x):,}" if pd.notna(x) else "")
        elif column in percent_columns:
            table[column] = table[column].map(lambda x: f"{x:,.2f}%" if pd.notna(x) else "")
        elif pd.api.types.is_numeric_dtype(table[column]):
            table[column] = table[column].map(lambda x: f"{x:,.{decimals}f}" if pd.notna(x) else "")

    display(table)


In [4]:
spark = (
    SparkSession.builder
    .appName("Retail Sales Performance Analysis")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## 1. Dataset Overview

The dataset contains supermarket transaction records covering:

- Transaction information
- Branch and city
- Customer type and gender
- Product line
- Unit price and quantity
- Sales and cost of goods sold (COGS)
- Gross income
- Payment method
- Customer rating
- Date and time

The analysis uses the original 1,000 transaction records without synthetic data generation or duplication.

## 2. Data Loading

The dataset is loaded from Google Drive into a PySpark DataFrame for analysis.


In [5]:
from google.colab import drive

drive.mount("/content/drive")

csv_path = "/content/drive/MyDrive/BigData/SuperMarket Analysis.csv"

print("Dataset path:")
print(csv_path)

Mounted at /content/drive
Dataset path:
/content/drive/MyDrive/BigData/SuperMarket Analysis.csv


In [6]:
df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [7]:
total_rows = df.count()
total_columns = len(df.columns)

print("=" * 60)
print("DATASET DIMENSIONS")
print("=" * 60)

print(f"Rows    : {total_rows:,}")
print(f"Columns : {total_columns}")

DATASET DIMENSIONS
Rows    : 1,000
Columns : 17


In [8]:
print("=" * 60)
print("DATASET SCHEMA")
print("=" * 60)

df.printSchema()

DATASET SCHEMA
root
 |-- Invoice ID: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer type: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Product line: string (nullable = true)
 |-- Unit price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Tax 5%: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Payment: string (nullable = true)
 |-- cogs: double (nullable = true)
 |-- gross margin percentage: double (nullable = true)
 |-- gross income: double (nullable = true)
 |-- Rating: double (nullable = true)



In [9]:
print("=" * 60)
print("DATASET PREVIEW")
print("=" * 60)

df.show(5, truncate=False)

DATASET PREVIEW
+-----------+------+---------+-------------+------+----------------------+----------+--------+-------+--------+---------+-----------+-----------+------+-----------------------+------------+------+
|Invoice ID |Branch|City     |Customer type|Gender|Product line          |Unit price|Quantity|Tax 5% |Sales   |Date     |Time       |Payment    |cogs  |gross margin percentage|gross income|Rating|
+-----------+------+---------+-------------+------+----------------------+----------+--------+-------+--------+---------+-----------+-----------+------+-----------------------+------------+------+
|750-67-8428|Alex  |Yangon   |Member       |Female|Health and beauty     |74.69     |7       |26.1415|548.9715|1/5/2019 |1:08:00 PM |Ewallet    |522.83|4.761904762            |26.1415     |9.1   |
|226-31-3081|Giza  |Naypyitaw|Normal       |Female|Electronic accessories|15.28     |5       |3.82   |80.22   |3/8/2019 |10:29:00 AM|Cash       |76.4  |4.761904762            |3.82        |9.6   |

In [10]:
total_transactions = df.count()
total_columns = len(df.columns)
total_cities = df.select("City").distinct().count()
total_product_lines = df.select("Product line").distinct().count()
total_branches = df.select("Branch").distinct().count()

overview = df.agg(
    F.sum("Sales").alias("Total Sales"),
    F.sum("Quantity").alias("Total Quantity"),
    F.avg("Rating").alias("Average Rating")
).first()

total_sales = overview["Total Sales"]
total_quantity = overview["Total Quantity"]
average_rating = overview["Average Rating"]

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Transactions       : {total_transactions:,}")
print(f"Columns            : {total_columns}")
print(f"Cities             : {total_cities}")
print(f"Product Lines      : {total_product_lines}")
print(f"Branches           : {total_branches}")

print("\n" + "=" * 60)
print("KEY BUSINESS METRICS")
print("=" * 60)

print(f"Total Sales        : {total_sales:,.2f}")
print(f"Total Quantity     : {total_quantity:,}")
print(f"Average Rating     : {average_rating:.2f}/10")

DATASET OVERVIEW
Transactions       : 1,000
Columns            : 17
Cities             : 3
Product Lines      : 6
Branches           : 3

KEY BUSINESS METRICS
Total Sales        : 322,966.75
Total Quantity     : 5,510
Average Rating     : 6.97/10


## 3. Data Quality Assessment

This section checks missing values and duplicate records before data preparation and analysis.


In [11]:
missing_values = (
    df.select([
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in df.columns
    ])
    .toPandas()
    .T
    .reset_index()
)

missing_values.columns = ["Column", "Missing Values"]

missing_values["Status"] = missing_values["Missing Values"].apply(
    lambda x: "OK" if x == 0 else "Needs Attention"
)

display(missing_values)

,Column,Missing Values,Status
0,Invoice ID,0,OK
1,Branch,0,OK
2,City,0,OK
3,Customer type,0,OK
4,Gender,0,OK
5,Product line,0,OK
6,Unit price,0,OK
7,Quantity,0,OK
8,Tax 5%,0,OK
9,Sales,0,OK


In [12]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - distinct_rows

duplicate_invoice_ids = (
    df.groupBy("Invoice ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)

print(f"Total rows                  : {total_rows:,}")
print(f"Distinct rows               : {distinct_rows:,}")
print(f"Exact duplicate rows        : {duplicate_rows:,}")
print(f"Duplicate Invoice IDs       : {duplicate_invoice_ids:,}")

if duplicate_rows == 0 and duplicate_invoice_ids == 0:
    print("Status                      : No duplicate records found.")
else:
    print("Status                      : Duplicate records detected.")

DUPLICATE CHECK
Total rows                  : 1,000
Distinct rows               : 1,000
Exact duplicate rows        : 0
Duplicate Invoice IDs       : 0
Status                      : No duplicate records found.


## 4. Data Preparation

The data preparation stage focuses on converting date and time fields into analytical features and creating derived metrics that can support sales and customer behavior analysis.

In [14]:
df = df.withColumn(
    "Date",
    F.to_date(F.col("Date"), "M/d/yyyy")
)

df = df.withColumn(
    "Month",
    F.date_format("Date", "MMM")
)

df = df.withColumn(
    "Month_Number",
    F.month("Date")
)

df = df.withColumn(
    "Day",
    F.dayofmonth("Date")
)

df = df.withColumn(
    "Day_Name",
    F.date_format("Date", "EEEE")
)

df = df.withColumn(
    "Time_Parsed",
    F.to_timestamp("Time", "h:mm:ss a")
)

df = df.withColumn(
    "Hour",
    F.hour("Time_Parsed")
)

df.select(
    "Date",
    "Month",
    "Month_Number",
    "Day",
    "Day_Name",
    "Time",
    "Hour"
).show(5, truncate=False)

+----------+-----+------------+---+--------+-----------+----+
|Date      |Month|Month_Number|Day|Day_Name|Time       |Hour|
+----------+-----+------------+---+--------+-----------+----+
|2019-01-05|Jan  |1           |5  |Saturday|1:08:00 PM |13  |
|2019-03-08|Mar  |3           |8  |Friday  |10:29:00 AM|10  |
|2019-03-03|Mar  |3           |3  |Sunday  |1:23:00 PM |13  |
|2019-01-27|Jan  |1           |27 |Sunday  |8:33:00 PM |20  |
|2019-02-08|Feb  |2           |8  |Friday  |10:37:00 AM|10  |
+----------+-----+------------+---+--------+-----------+----+
only showing top 5 rows


In [15]:
df = df.withColumn(
    "Gross Margin",
    F.when(
        F.col("Sales") != 0,
        F.col("gross income") / F.col("Sales") * 100
    ).otherwise(0)
)

df.select(
    "Invoice ID",
    "Sales",
    "gross income",
    "Gross Margin"
).show(5, truncate=False)

+-----------+--------+------------+-----------------+
|Invoice ID |Sales   |gross income|Gross Margin     |
+-----------+--------+------------+-----------------+
|750-67-8428|548.9715|26.1415     |4.761904761904763|
|226-31-3081|80.22   |3.82        |4.761904761904762|
|631-41-3108|340.5255|16.2155     |4.761904761904761|
|123-19-1176|489.048 |23.288      |4.761904761904762|
|373-73-7910|634.3785|30.2085     |4.761904761904762|
+-----------+--------+------------+-----------------+
only showing top 5 rows


## 5. Descriptive Statistics

Before performing detailed analysis, descriptive statistics are used to understand the distribution and range of key numerical variables.

In [16]:
numeric_columns = [
    "Unit price",
    "Quantity",
    "Tax 5%",
    "Sales",
    "cogs",
    "gross income",
    "Rating"
]

descriptive_stats = (
    df.select(numeric_columns)
    .describe()
    .toPandas()
)

display(descriptive_stats)

,summary,Unit price,Quantity,Tax 5%,Sales,cogs,gross income,Rating
0,count,1000,1000,1000,1000,1000,1000,1000
1,mean,55.67212999999998,5.51,15.379369000000002,322.96674900000005,307.58738000000034,15.379369000000002,6.972700000000003
2,stddev,26.494628347919786,2.9234305954556956,11.70882548099866,245.8853351009718,234.17650961997333,11.70882548099866,1.7185802943791213
3,min,10.08,1,0.5085,10.6785,10.17,0.5085,4.0
4,max,99.96,10,49.65,1042.65,993.0,49.65,10.0


## 6. Key Performance Indicators

The following KPIs summarize the overall business performance of the 1,000 transactions in the dataset.

In [17]:
kpi = df.agg(
    F.sum("Sales").alias("Total Sales"),
    F.countDistinct("Invoice ID").alias("Total Transactions"),
    F.sum("Quantity").alias("Total Quantity"),
    F.avg("Sales").alias("Average Transaction Value"),
    F.avg("Rating").alias("Average Rating"),
    F.sum("gross income").alias("Total Gross Income")
).first()

total_sales = kpi["Total Sales"]
total_transactions = kpi["Total Transactions"]
total_quantity = kpi["Total Quantity"]
average_transaction_value = kpi["Average Transaction Value"]
average_rating = kpi["Average Rating"]
total_gross_income = kpi["Total Gross Income"]

print("=" * 60)
print("KEY PERFORMANCE INDICATORS")
print("=" * 60)

print(f"Total Sales               : {total_sales:,.2f}")
print(f"Total Transactions        : {total_transactions:,}")
print(f"Total Quantity Sold       : {total_quantity:,}")
print(f"Average Transaction Value : {average_transaction_value:,.2f}")
print(f"Average Rating            : {average_rating:.2f}/10")
print(f"Total Gross Income        : {total_gross_income:,.2f}")

KEY PERFORMANCE INDICATORS
Total Sales               : 322,966.75
Total Transactions        : 1,000
Total Quantity Sold       : 5,510
Average Transaction Value : 322.97
Average Rating            : 6.97/10
Total Gross Income        : 15,379.37


In [18]:
kpi_labels = [
    "Total Sales",
    "Transactions",
    "Quantity Sold",
    "Avg. Transaction",
    "Avg. Rating",
    "Gross Income"
]

kpi_values = [
    total_sales,
    total_transactions,
    total_quantity,
    average_transaction_value,
    average_rating,
    total_gross_income
]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [{"type": "indicator"}] * 3,
        [{"type": "indicator"}] * 3
    ]
)

for i, (label, value) in enumerate(zip(kpi_labels, kpi_values)):

    row = 1 if i < 3 else 2
    col = (i % 3) + 1

    fig.add_trace(
        go.Indicator(
            mode="number",
            value=value,
            title={"text": label}
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="Retail Sales KPI Overview",
    height=500,
    template="plotly_white"
)

fig.show()

## 7. Sales Performance Analysis

This section evaluates sales performance across cities and product lines to identify the strongest contributors to overall revenue.

In [19]:
sales_city = (
    df.groupBy("City")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.avg("Sales").alias("Average Transaction")
    )
    .orderBy(F.desc("Total Sales"))
    .toPandas()
)

sales_city["Sales Contribution %"] = (
    sales_city["Total Sales"]
    / sales_city["Total Sales"].sum()
    * 100
)

display_formatted(
    sales_city,
    integer_columns=["Transactions", "Quantity Sold"],
    percent_columns=["Sales Contribution %"]
)

,City,Total Sales,Transactions,Quantity Sold,Average Transaction,Sales Contribution %
0,Naypyitaw,"110,568.71",328,"1,831",337.10,34.24%
1,Yangon,"106,200.37",340,"1,859",312.35,32.88%
2,Mandalay,"106,197.67",332,"1,820",319.87,32.88%


In [20]:
fig = px.bar(
    sales_city.sort_values("Total Sales"),
    x="Total Sales",
    y="City",
    orientation="h",
    text="Total Sales",
    title="Total Sales by City",
    labels={
        "Total Sales": "Total Sales",
        "City": "City"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_tickformat=",.0f",
    yaxis_title="",
    height=450
)

fig.show()

In [21]:
sales_product = (
    df.groupBy("Product line")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.sum("gross income").alias("Gross Income"),
        F.avg("Rating").alias("Average Rating"),
        F.countDistinct("Invoice ID").alias("Transactions")
    )
    .orderBy(F.desc("Total Sales"))
    .toPandas()
)

sales_product["Sales Contribution %"] = (
    sales_product["Total Sales"]
    / sales_product["Total Sales"].sum()
    * 100
)

display_formatted(
    sales_product,
    integer_columns=["Transactions", "Quantity Sold"],
    percent_columns=["Sales Contribution %"]
)

,Product line,Total Sales,Quantity Sold,Gross Income,Average Rating,Transactions,Sales Contribution %
0,Food and beverages,"56,144.84",952,"2,673.56",7.11,174,17.38%
1,Sports and travel,"55,122.83",920,"2,624.90",6.92,166,17.07%
2,Electronic accessories,"54,337.53",971,"2,587.50",6.92,170,16.82%
3,Fashion accessories,"54,305.90",902,"2,586.00",7.03,178,16.81%
4,Home and lifestyle,"53,861.91",911,"2,564.85",6.84,160,16.68%
5,Health and beauty,"49,193.74",854,"2,342.56",7.00,152,15.23%


In [22]:
fig = px.bar(
    sales_product.sort_values("Total Sales"),
    x="Total Sales",
    y="Product line",
    orientation="h",
    text="Total Sales",
    title="Sales Performance by Product Line",
    labels={
        "Total Sales": "Total Sales",
        "Product line": "Product Line"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_tickformat=",.0f",
    yaxis_title="",
    height=500
)

fig.show()

In [23]:
profitability = (
    df.groupBy("Product line")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.sum("cogs").alias("Total COGS"),
        F.sum("gross income").alias("Gross Income"),
        F.avg("gross margin percentage").alias("Gross Margin %")
    )
    .orderBy(F.desc("Gross Income"))
    .toPandas()
)

display_formatted(profitability, percent_columns=["Gross Margin %"])

,Product line,Total Sales,Total COGS,Gross Income,Gross Margin %
0,Food and beverages,"56,144.84","53,471.28","2,673.56",4.76%
1,Sports and travel,"55,122.83","52,497.93","2,624.90",4.76%
2,Electronic accessories,"54,337.53","51,750.03","2,587.50",4.76%
3,Fashion accessories,"54,305.89","51,719.90","2,585.99",4.76%
4,Home and lifestyle,"53,861.91","51,297.06","2,564.85",4.76%
5,Health and beauty,"49,193.74","46,851.18","2,342.56",4.76%


In [24]:
fig = px.bar(
    profitability.sort_values("Gross Income"),
    x="Gross Income",
    y="Product line",
    orientation="h",
    text="Gross Income",
    title="Gross Income by Product Line",
    labels={
        "Gross Income": "Gross Income",
        "Product line": "Product Line"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_tickformat=",.0f",
    yaxis_title="",
    height=500
)

fig.show()

## 8. Customer Analysis

This section examines customer segments to understand differences in transaction volume, sales contribution, quantity purchased, and average transaction value.

In [25]:
customer_analysis = (
    df.groupBy("Customer type")
    .agg(
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Sales").alias("Total Sales"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.avg("Sales").alias("Average Transaction"),
        F.avg("Rating").alias("Average Rating")
    )
    .orderBy(F.desc("Total Sales"))
    .toPandas()
)

customer_analysis["Sales Contribution %"] = (
    customer_analysis["Total Sales"]
    / customer_analysis["Total Sales"].sum()
    * 100
)

display_formatted(
    customer_analysis,
    integer_columns=["Transactions", "Quantity Sold"],
    percent_columns=["Sales Contribution %"]
)

,Customer type,Transactions,Total Sales,Quantity Sold,Average Transaction,Average Rating,Sales Contribution %
0,Member,565,"189,694.76","3,181",335.74,6.92,58.74%
1,Normal,435,"133,271.99","2,329",306.37,7.04,41.26%


In [26]:
fig = px.bar(
    customer_analysis,
    x="Customer type",
    y="Total Sales",
    text="Total Sales",
    title="Sales by Customer Type",
    labels={
        "Customer type": "Customer Type",
        "Total Sales": "Total Sales"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    yaxis_tickformat=",.0f",
    height=450
)

fig.show()

In [27]:
gender_analysis = (
    df.groupBy("Gender")
    .agg(
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Sales").alias("Total Sales"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.avg("Sales").alias("Average Transaction"),
        F.avg("Rating").alias("Average Rating")
    )
    .orderBy(F.desc("Total Sales"))
    .toPandas()
)

gender_analysis["Sales Contribution %"] = (
    gender_analysis["Total Sales"]
    / gender_analysis["Total Sales"].sum()
    * 100
)

display_formatted(
    gender_analysis,
    integer_columns=["Transactions", "Quantity Sold"],
    percent_columns=["Sales Contribution %"]
)

,Gender,Transactions,Total Sales,Quantity Sold,Average Transaction,Average Rating,Sales Contribution %
0,Female,571,"194,671.84","3,288",340.93,6.96,60.28%
1,Male,429,"128,294.91","2,222",299.06,6.99,39.72%


In [28]:
fig = px.bar(
    gender_analysis,
    x="Gender",
    y="Total Sales",
    text="Total Sales",
    title="Sales by Gender",
    labels={
        "Gender": "Gender",
        "Total Sales": "Total Sales"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    yaxis_tickformat=",.0f",
    height=450
)

fig.show()

## 9. Payment Method Analysis

This section evaluates payment preferences based on transaction volume, sales contribution, and average transaction value.

In [29]:
payment_analysis = (
    df.groupBy("Payment")
    .agg(
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Sales").alias("Total Sales"),
        F.avg("Sales").alias("Average Transaction")
    )
    .orderBy(F.desc("Transactions"))
    .toPandas()
)

payment_analysis["Transaction Share %"] = (
    payment_analysis["Transactions"]
    / payment_analysis["Transactions"].sum()
    * 100
)

payment_analysis["Sales Contribution %"] = (
    payment_analysis["Total Sales"]
    / payment_analysis["Total Sales"].sum()
    * 100
)

display_formatted(
    payment_analysis,
    integer_columns=["Transactions"],
    percent_columns=["Transaction Share %", "Sales Contribution %"]
)

,Payment,Transactions,Total Sales,Average Transaction,Transaction Share %,Sales Contribution %
0,Ewallet,345,"109,993.11",318.82,34.50%,34.06%
1,Cash,344,"112,206.57",326.18,34.40%,34.74%
2,Credit card,311,"100,767.07",324.01,31.10%,31.20%


In [30]:
fig = px.pie(
    payment_analysis,
    names="Payment",
    values="Transactions",
    hole=0.45,
    title="Transaction Distribution by Payment Method",
    template="plotly_white"
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label"
)

fig.update_layout(
    height=450
)

fig.show()

### Payment Insight

Ewallet is the most frequently used payment method based on transaction count, while Cash generates the highest total sales. This distinction shows that the most popular payment method is not necessarily the payment method contributing the highest sales value.

## 10. Time-Based Analysis

This section analyzes sales patterns over time to identify daily, monthly, and hourly sales behavior.

In [31]:
daily_sales = (
    df.groupBy("Date")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.avg("Sales").alias("Average Transaction")
    )
    .orderBy("Date")
    .toPandas()
)

display_formatted(daily_sales, integer_columns=["Transactions", "Quantity Sold"])

,Date,Total Sales,Transactions,Quantity Sold,Average Transaction
0,2019-01-01,"4,745.18",12,81,395.43
1,2019-01-02,"1,945.50",8,48,243.19
2,2019-01-03,"2,078.13",8,37,259.77
3,2019-01-04,"1,623.69",6,32,270.61
4,2019-01-05,"3,536.68",12,55,294.72
...,...,...,...,...,...
84,2019-03-26,"1,962.51",13,52,150.96
85,2019-03-27,"2,902.82",10,45,290.28
86,2019-03-28,"2,229.40",10,48,222.94
87,2019-03-29,"4,023.24",8,54,502.91


In [32]:
fig = px.line(
    daily_sales,
    x="Date",
    y="Total Sales",
    markers=True,
    title="Daily Sales Trend",
    labels={
        "Date": "Date",
        "Total Sales": "Total Sales"
    },
    template="plotly_white"
)

fig.update_layout(
    yaxis_tickformat=",.0f",
    height=450
)

fig.show()

In [33]:
monthly_sales = (
    df.groupBy("Month_Number", "Month")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.sum("gross income").alias("Gross Income")
    )
    .orderBy("Month_Number")
    .toPandas()
)

monthly_sales["Sales Growth %"] = (
    monthly_sales["Total Sales"]
    .pct_change() * 100
)

display_formatted(monthly_sales, integer_columns=["Transactions", "Quantity Sold"], percent_columns=["Sales Growth %"])

,Month_Number,Month,Total Sales,Transactions,Quantity Sold,Gross Income,Sales Growth %
0,1.00,Jan,"116,291.87",352,"1,965","5,537.71",
1,2.00,Feb,"97,219.37",303,"1,654","4,629.49",-16.40%
2,3.00,Mar,"109,455.51",345,"1,891","5,212.17",12.59%


In [34]:
fig = px.line(
    monthly_sales,
    x="Month",
    y="Total Sales",
    markers=True,
    text="Total Sales",
    title="Monthly Sales Trend",
    labels={
        "Month": "Month",
        "Total Sales": "Total Sales"
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="top center"
)

fig.update_layout(
    yaxis_tickformat=",.0f",
    height=450
)

fig.show()

In [35]:
hourly_sales = (
    df.groupBy("Hour")
    .agg(
        F.sum("Sales").alias("Total Sales"),
        F.countDistinct("Invoice ID").alias("Transactions"),
        F.sum("Quantity").alias("Quantity Sold"),
        F.avg("Sales").alias("Average Transaction")
    )
    .orderBy("Hour")
    .toPandas()
)

display_formatted(hourly_sales, integer_columns=["Hour", "Transactions", "Quantity Sold"])

,Hour,Total Sales,Transactions,Quantity Sold,Average Transaction
0,10,"31,421.48",101,525,311.10
1,11,"30,377.33",90,513,337.53
2,12,"26,065.88",89,501,292.88
3,13,"34,723.23",103,585,337.12
4,14,"30,828.40",83,495,371.43
5,15,"31,179.51",102,530,305.68
6,16,"25,226.32",77,420,327.61
7,17,"24,445.22",74,415,330.34
8,18,"26,030.34",93,475,279.90
9,19,"39,699.51",113,649,351.32


In [36]:
fig = px.line(
    hourly_sales,
    x="Hour",
    y="Total Sales",
    markers=True,
    title="Sales Performance by Hour",
    labels={
        "Hour": "Hour of Day",
        "Total Sales": "Total Sales"
    },
    template="plotly_white"
)

fig.update_layout(
    xaxis=dict(
        dtick=1
    ),
    yaxis_tickformat=",.0f",
    height=450
)

fig.show()

In [37]:
peak_hour = (
    hourly_sales
    .sort_values("Total Sales", ascending=False)
    .iloc[0]
)

print("=" * 60)
print("PEAK SALES HOUR")
print("=" * 60)

print(
    f"Peak sales hour : {int(peak_hour['Hour']):02d}:00"
)

print(
    f"Total sales     : {peak_hour['Total Sales']:,.2f}"
)

print(
    f"Transactions    : {int(peak_hour['Transactions']):,}"
)

PEAK SALES HOUR
Peak sales hour : 19:00
Total sales     : 39,699.51
Transactions    : 113


In [38]:
top_transactions = (
    df.select(
        "Invoice ID",
        "City",
        "Product line",
        "Customer type",
        "Payment",
        "Quantity",
        "Sales",
        "gross income",
        "Rating"
    )
    .orderBy(F.desc("Sales"))
    .limit(5)
    .toPandas()
)

display_formatted(top_transactions, integer_columns=["Quantity"])

,Invoice ID,City,Product line,Customer type,Payment,Quantity,Sales,gross income,Rating
0,860-79-0874,Naypyitaw,Fashion accessories,Member,Credit card,10,"1,042.65",49.65,6.60
1,687-47-8271,Yangon,Fashion accessories,Normal,Credit card,10,"1,039.29",49.49,8.70
2,283-26-5248,Naypyitaw,Food and beverages,Member,Ewallet,10,"1,034.46",49.26,4.50
3,751-41-9720,Naypyitaw,Home and lifestyle,Normal,Ewallet,10,"1,023.75",48.75,8.00
4,303-96-2227,Mandalay,Home and lifestyle,Normal,Ewallet,10,"1,022.49",48.69,4.40


In [39]:
sales_rating = (
    df.select("Sales", "Rating")
    .toPandas()
)

correlation = sales_rating["Sales"].corr(
    sales_rating["Rating"]
)

print(
    f"Correlation between Sales and Rating: {correlation:.3f}"
)

Correlation between Sales and Rating: -0.036


In [40]:
if abs(correlation) < 0.10:
    relationship = "negligible"
elif abs(correlation) < 0.30:
    relationship = "weak"
elif abs(correlation) < 0.50:
    relationship = "moderate"
else:
    relationship = "strong"

direction = "positive" if correlation > 0 else "negative"

print(
    f"The correlation coefficient of {correlation:.3f} "
    f"indicates a {relationship} {direction} linear relationship "
    f"between customer rating and sales."
)

print(
    "This suggests that customer rating does not have a meaningful "
    "linear association with transaction sales in this dataset."
)

The correlation coefficient of -0.036 indicates a negligible negative linear relationship between customer rating and sales.
This suggests that customer rating does not have a meaningful linear association with transaction sales in this dataset.


In [41]:
fig = px.scatter(
    sales_rating,
    x="Rating",
    y="Sales",
    title="Relationship Between Customer Rating and Sales",
    labels={
        "Rating": "Customer Rating",
        "Sales": "Sales"
    },
    trendline="ols",
    template="plotly_white"
)

fig.update_layout(
    yaxis_tickformat=",.0f",
    height=500
)

fig.show()

## 11. Key Business Insights

The analysis results are summarized into key findings that are directly supported by the transaction data.

In [42]:
top_city = (
    sales_city
    .sort_values("Total Sales", ascending=False)
    .iloc[0]
)

top_product = (
    sales_product
    .sort_values("Total Sales", ascending=False)
    .iloc[0]
)

top_customer = (
    customer_analysis
    .sort_values("Total Sales", ascending=False)
    .iloc[0]
)

most_used_payment = (
    payment_analysis
    .sort_values("Transactions", ascending=False)
    .iloc[0]
)

highest_sales_payment = (
    payment_analysis
    .sort_values("Total Sales", ascending=False)
    .iloc[0]
)

highest_gross_income_product = (
    sales_product
    .sort_values("Gross Income", ascending=False)
    .iloc[0]
)

highest_rating_product = (
    sales_product
    .sort_values("Average Rating", ascending=False)
    .iloc[0]
)

print("=" * 70)
print("KEY BUSINESS INSIGHTS")
print("=" * 70)

print(
    f"\n1. CITY PERFORMANCE\n"
    f"{top_city['City']} generated the highest total sales of "
    f"{top_city['Total Sales']:,.2f}."
)

print(
    f"\n2. PRODUCT PERFORMANCE\n"
    f"{top_product['Product line']} was the highest-selling product line, "
    f"generating {top_product['Total Sales']:,.2f}."
)

print(
    f"\n3. CUSTOMER SEGMENT\n"
    f"{top_customer['Customer type']} customers generated the highest "
    f"sales at {top_customer['Total Sales']:,.2f}."
)

print(
    f"\n4. PAYMENT BEHAVIOR\n"
    f"{most_used_payment['Payment']} was the most frequently used payment "
    f"method with {int(most_used_payment['Transactions']):,} transactions."
)

print(
    f"\n5. PAYMENT SALES CONTRIBUTION\n"
    f"{highest_sales_payment['Payment']} generated the highest total "
    f"sales of {highest_sales_payment['Total Sales']:,.2f}."
)

print(
    f"\n6. PROFITABILITY\n"
    f"{highest_gross_income_product['Product line']} generated the highest "
    f"gross income of {highest_gross_income_product['Gross Income']:,.2f}."
)

print(
    f"\n7. CUSTOMER EXPERIENCE\n"
    f"{highest_rating_product['Product line']} recorded the highest "
    f"average rating of {highest_rating_product['Average Rating']:.2f}/10."
)

print(
    f"\n8. SALES AND RATING RELATIONSHIP\n"
    f"The correlation between sales and customer rating was "
    f"{correlation:.3f}, indicating a negligible linear relationship."
)

print(
    f"\n9. PEAK SALES HOUR\n"
    f"The highest sales were recorded at approximately "
    f"{int(peak_hour['Hour']):02d}:00."
)

KEY BUSINESS INSIGHTS

1. CITY PERFORMANCE
Naypyitaw generated the highest total sales of 110,568.71.

2. PRODUCT PERFORMANCE
Food and beverages was the highest-selling product line, generating 56,144.84.

3. CUSTOMER SEGMENT
Member customers generated the highest sales at 189,694.76.

4. PAYMENT BEHAVIOR
Ewallet was the most frequently used payment method with 345 transactions.

5. PAYMENT SALES CONTRIBUTION
Cash generated the highest total sales of 112,206.57.

6. PROFITABILITY
Food and beverages generated the highest gross income of 2,673.56.

7. CUSTOMER EXPERIENCE
Food and beverages recorded the highest average rating of 7.11/10.

8. SALES AND RATING RELATIONSHIP
The correlation between sales and customer rating was -0.036, indicating a negligible linear relationship.

9. PEAK SALES HOUR
The highest sales were recorded at approximately 19:00.


## 12. Business Recommendations

The recommendations below are derived from the observed sales, profitability, customer, payment, and time-based patterns.

In [43]:
recommendations = pd.DataFrame([
    {
        "Finding": f"{top_city['City']} generated the highest total sales ({top_city['Total Sales']:,.2f}).",
        "Implication": "Demand is relatively stronger in this location, making availability and execution more important.",
        "Action": f"Prioritize inventory availability and targeted promotions in {top_city['City']}."
    },
    {
        "Finding": f"{top_product['Product line']} was the highest-selling product line ({top_product['Total Sales']:,.2f}).",
        "Implication": "This category is a major contributor to revenue and may have stronger customer demand.",
        "Action": f"Prioritize stock planning and monitor stock-out risk for {top_product['Product line']}."
    },
    {
        "Finding": f"{top_customer['Customer type']} customers generated the highest sales ({top_customer['Total Sales']:,.2f}).",
        "Implication": "This customer segment represents an important source of revenue.",
        "Action": "Strengthen retention and loyalty initiatives while monitoring segment-level transaction value."
    },
    {
        "Finding": f"{most_used_payment['Payment']} was the most frequently used payment method ({int(most_used_payment['Transactions']):,} transactions).",
        "Implication": "Customers show a strong preference for this payment method by transaction frequency.",
        "Action": f"Maintain reliable {most_used_payment['Payment']} payment availability and monitor transaction success rates."
    },
    {
        "Finding": f"{highest_sales_payment['Payment']} generated the highest sales value ({highest_sales_payment['Total Sales']:,.2f}).",
        "Implication": "Payment frequency and payment value tell different stories in this dataset.",
        "Action": "Monitor both transaction count and transaction value when evaluating payment performance."
    },
    {
        "Finding": f"Sales peaked at approximately {int(peak_hour['Hour']):02d}:00.",
        "Implication": "Demand is concentrated during specific hours, increasing the importance of operational readiness.",
        "Action": f"Align staffing, inventory readiness, and promotional timing around the {int(peak_hour['Hour']):02d}:00 peak period."
    },
    {
        "Finding": f"The sales-rating correlation was {correlation:.3f}, indicating a negligible linear relationship.",
        "Implication": "Customer rating should not be treated as a strong linear predictor of transaction sales from this dataset alone.",
        "Action": "Monitor rating alongside sales as a customer-experience KPI without assuming a direct causal effect."
    }
])

display(recommendations)


,Finding,Implication,Action
0,Naypyitaw generated the highest total sales (1...,Demand is relatively stronger in this location...,Prioritize inventory availability and targeted...
1,Food and beverages was the highest-selling pro...,This category is a major contributor to revenu...,Prioritize stock planning and monitor stock-ou...
2,Member customers generated the highest sales (...,This customer segment represents an important ...,Strengthen retention and loyalty initiatives w...
3,Ewallet was the most frequently used payment m...,Customers show a strong preference for this pa...,Maintain reliable Ewallet payment availability...
4,"Cash generated the highest sales value (112,20...",Payment frequency and payment value tell diffe...,Monitor both transaction count and transaction...
5,Sales peaked at approximately 19:00.,"Demand is concentrated during specific hours, ...","Align staffing, inventory readiness, and promo..."
6,"The sales-rating correlation was -0.036, indic...",Customer rating should not be treated as a str...,Monitor rating alongside sales as a customer-e...


## 13. Executive Dashboard

The dashboard provides a concise overview of the most important sales and customer performance indicators for management-level review.

In [44]:
dashboard_kpi = {
    "Total Sales": total_sales,
    "Transactions": total_transactions,
    "Quantity Sold": total_quantity,
    "Avg. Transaction": average_transaction_value,
    "Avg. Rating": average_rating,
    "Gross Income": total_gross_income
}

sales_city_dashboard = sales_city.copy()
product_dashboard = sales_product.copy()
monthly_dashboard = monthly_sales.copy()
payment_dashboard = payment_analysis.copy()
hourly_dashboard = hourly_sales.copy()


In [46]:
fig = make_subplots(

    rows=4,
    cols=2,

    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "scatter"}],
        [{"type": "bar"}, {"type": "bar"}]
    ],

    subplot_titles=(
        "Total Sales",
        "Gross Income",
        "Sales by City",
        "Sales by Product Line",
        "Monthly Sales Trend",
        "Hourly Sales Trend",
        "Payment Method Usage",
        "Transaction Volume by Customer Type"
    ),

    vertical_spacing=0.07,
    row_heights=[0.16, 0.27, 0.28, 0.29]
)


fig.add_trace(

    go.Indicator(

        mode="number",

        value=dashboard_kpi["Total Sales"],

        number={
            "valueformat": ",.2f",
            "font": {
                "size": 36
            }
        }

    ),

    row=1,
    col=1
)


fig.add_trace(

    go.Indicator(

        mode="number",

        value=dashboard_kpi["Gross Income"],

        number={
            "valueformat": ",.2f",
            "font": {
                "size": 36
            }
        }

    ),

    row=1,
    col=2
)


fig.add_trace(

    go.Bar(

        x=sales_city_dashboard["City"],

        y=sales_city_dashboard["Total Sales"],

        text=sales_city_dashboard["Total Sales"],

        texttemplate="%{text:,.0f}",

        textposition="outside",

        cliponaxis=False,

        hovertemplate=(
            "<b>%{x}</b><br>"
            "Sales: %{y:,.2f}"
            "<extra></extra>"
        )

    ),

    row=2,
    col=1
)


fig.add_trace(

    go.Bar(

        x=product_dashboard["Product line"],

        y=product_dashboard["Total Sales"],

        text=product_dashboard["Total Sales"],

        texttemplate="%{text:,.0f}",

        textposition="outside",

        cliponaxis=False,

        hovertemplate=(
            "<b>%{x}</b><br>"
            "Sales: %{y:,.2f}"
            "<extra></extra>"
        )

    ),

    row=2,
    col=2
)


fig.add_trace(

    go.Scatter(

        x=monthly_dashboard["Month"],

        y=monthly_dashboard["Total Sales"],

        mode="lines+markers",

        line={
            "width": 3
        },

        marker={
            "size": 8
        },

        hovertemplate=(
            "<b>%{x}</b><br>"
            "Sales: %{y:,.2f}"
            "<extra></extra>"
        )

    ),

    row=3,
    col=1
)


fig.add_trace(

    go.Scatter(

        x=hourly_dashboard["Hour"],

        y=hourly_dashboard["Total Sales"],

        mode="lines+markers",

        line={
            "width": 3
        },

        marker={
            "size": 8
        },

        hovertemplate=(
            "<b>Hour %{x}:00</b><br>"
            "Sales: %{y:,.2f}"
            "<extra></extra>"
        )

    ),

    row=3,
    col=2
)


fig.add_trace(

    go.Bar(

        x=payment_dashboard["Payment"],

        y=payment_dashboard["Transactions"],

        text=payment_dashboard["Transactions"],

        texttemplate="%{text:,.0f}",

        textposition="outside",

        cliponaxis=False,

        hovertemplate=(
            "<b>%{x}</b><br>"
            "Transactions: %{y:,.0f}"
            "<extra></extra>"
        )

    ),

    row=4,
    col=1
)


fig.add_trace(

    go.Bar(

        x=customer_analysis["Customer type"],

        y=customer_analysis["Transactions"],

        text=customer_analysis["Transactions"],

        texttemplate="%{text:,.0f}",

        textposition="outside",

        cliponaxis=False,

        hovertemplate=(
            "<b>%{x}</b><br>"
            "Transactions: %{y:,.0f}"
            "<extra></extra>"
        )

    ),

    row=4,
    col=2
)


fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=2,
    col=1
)

fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=2,
    col=2
)

fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=3,
    col=1
)

fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=3,
    col=2
)

fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=4,
    col=1
)

fig.update_yaxes(

    tickformat=",.0f",

    automargin=True,

    row=4,
    col=2
)


fig.update_xaxes(

    tickangle=-35,

    automargin=True,

    row=2,
    col=2
)


fig.update_xaxes(

    automargin=True,

    row=3,
    col=1
)


fig.update_xaxes(

    dtick=1,

    tickformat="02d",

    automargin=True,

    row=3,
    col=2
)


fig.update_xaxes(

    automargin=True,

    row=4,
    col=1
)


fig.update_xaxes(

    automargin=True,

    row=4,
    col=2
)


fig.update_layout(

    title={
        "text": "Retail Sales Performance Dashboard",
        "x": 0.5,
        "xanchor": "center",
        "font": {
            "size": 28
        }
    },

    height=1500,

    width=1500,

    showlegend=False,

    template="plotly_white",

    margin={
        "l": 80,
        "r": 80,
        "t": 110,
        "b": 100
    },

    font={
        "size": 14
    },

    hoverlabel={
        "font_size": 13
    }
)


for annotation in fig.layout.annotations:

    annotation.font = {
        "size": 18
    }

fig.show()

## 14. Conclusion

The analysis summarizes the overall sales performance and identifies the main business opportunities observed in the dataset.

In [47]:
print("=" * 70)
print("CONCLUSION")
print("=" * 70)

print(
    f"\nThe analysis of {total_transactions:,} supermarket transactions "
    f"shows that {top_city['City']} was the highest-performing city, "
    f"generating {top_city['Total Sales']:,.2f} in sales."
)

print(
    f"\n{top_product['Product line']} was the strongest product line "
    f"based on total sales, generating {top_product['Total Sales']:,.2f}."
)

print(
    f"\n{top_customer['Customer type']} customers contributed the highest "
    f"sales among customer segments, with "
    f"{top_customer['Total Sales']:,.2f}."
)

print(
    f"\nEwallet was the most frequently used payment method with "
    f"{int(most_used_payment['Transactions']):,} transactions, while "
    f"{highest_sales_payment['Payment']} generated the highest sales value."
)

print(
    f"\nThe correlation between customer rating and sales was "
    f"{correlation:.3f}, indicating a negligible linear relationship."
)

print(
    f"\nOverall, the findings suggest that management should focus on "
    f"high-performing locations, strong product categories, customer "
    f"retention, payment availability, and peak sales periods."
)

CONCLUSION

The analysis of 1,000 supermarket transactions shows that Naypyitaw was the highest-performing city, generating 110,568.71 in sales.

Food and beverages was the strongest product line based on total sales, generating 56,144.84.

Member customers contributed the highest sales among customer segments, with 189,694.76.

Ewallet was the most frequently used payment method with 345 transactions, while Cash generated the highest sales value.

The correlation between customer rating and sales was -0.036, indicating a negligible linear relationship.

Overall, the findings suggest that management should focus on high-performing locations, strong product categories, customer retention, payment availability, and peak sales periods.


## 15. Project Takeaways

### Analytical Skills Demonstrated

- Data loading and processing using PySpark
- Data quality assessment
- Data transformation and feature engineering
- Exploratory data analysis and KPI development
- Sales, profitability, customer, payment, and time-based analysis
- Correlation analysis and statistical interpretation
- Interactive visualization using Plotly
- Business insight generation and data-driven recommendations
- Executive dashboard development

### Tools

**Python | PySpark | Pandas | Plotly | Google Colab**

### Dataset

Original supermarket transaction dataset containing 1,000 transactions.


## 16. Project Summary

A concise management-level summary of the analysis results and business focus areas.


In [48]:
print("=" * 80)
print("RETAIL SALES PERFORMANCE ANALYSIS — PROJECT SUMMARY")
print("=" * 80)

print("\nDATASET")
print(f"- Transactions analyzed : {total_transactions:,}")
print(f"- Cities                : {df.select('City').distinct().count()}")
print(f"- Product lines         : {df.select('Product line').distinct().count()}")
print(f"- Original dataset columns : 17")
print(f"- Analysis columns       : {len(df.columns)}")

print("\nKEY PERFORMANCE INDICATORS")
print(f"- Total Sales           : {total_sales:,.2f}")
print(f"- Total Quantity Sold   : {total_quantity:,}")
print(f"- Average Transaction   : {average_transaction_value:,.2f}")
print(f"- Average Rating        : {average_rating:.2f}/10")
print(f"- Total Gross Income    : {total_gross_income:,.2f}")

print("\nTOP PERFORMERS")
print(f"- Top City              : {top_city['City']}")
print(f"- Top Product Line      : {top_product['Product line']}")
print(f"- Top Customer Type     : {top_customer['Customer type']}")
print(f"- Most Used Payment     : {most_used_payment['Payment']}")
print(f"- Highest Sales Payment : {highest_sales_payment['Payment']}")
print(
    f"- Top Gross Income Product : "
    f"{highest_gross_income_product['Product line']}"
)

print("\nKEY FINDINGS")
print(
    f"- Highest city sales    : {top_city['Total Sales']:,.2f}"
)

print(
    f"- Highest product sales : {top_product['Total Sales']:,.2f}"
)

print(
    f"- Most used payment     : "
    f"{most_used_payment['Payment']} "
    f"({int(most_used_payment['Transactions']):,} transactions)"
)

print(
    f"- Sales-rating correlation : {correlation:.3f}"
)

print(
    f"- Peak sales hour       : "
    f"{int(peak_hour['Hour']):02d}:00"
)

print("\nBUSINESS FOCUS")
print(
    "- Prioritize high-performing locations and product categories."
)

print(
    "- Strengthen customer retention strategies."
)

print(
    "- Maintain reliable payment infrastructure, particularly for "
    "frequently used payment methods."
)

print(
    "- Align inventory and staffing with observed sales patterns."
)

print(
    "- Monitor customer experience without assuming a direct causal "
    "relationship between ratings and sales."
)

RETAIL SALES PERFORMANCE ANALYSIS — PROJECT SUMMARY

DATASET
- Transactions analyzed : 1,000
- Cities                : 3
- Product lines         : 6
- Original dataset columns : 17
- Analysis columns       : 24

KEY PERFORMANCE INDICATORS
- Total Sales           : 322,966.75
- Total Quantity Sold   : 5,510
- Average Transaction   : 322.97
- Average Rating        : 6.97/10
- Total Gross Income    : 15,379.37

TOP PERFORMERS
- Top City              : Naypyitaw
- Top Product Line      : Food and beverages
- Top Customer Type     : Member
- Most Used Payment     : Ewallet
- Highest Sales Payment : Cash
- Top Gross Income Product : Food and beverages

KEY FINDINGS
- Highest city sales    : 110,568.71
- Highest product sales : 56,144.84
- Most used payment     : Ewallet (345 transactions)
- Sales-rating correlation : -0.036
- Peak sales hour       : 19:00

BUSINESS FOCUS
- Prioritize high-performing locations and product categories.
- Strengthen customer retention strategies.
- Maintain reli